# Appendix: A Classic Text Baseline

**Candidate Name:** Matteo Antony Mistretta

**Date:** 2026-10-17

**Environment:** Python 3.14, scikit-learn 1.9, PyTorch 2.14 (CPU) (exact versions in `requirements.txt`).

*An extra experiment, not one of the bonus options: it answers a question that came up while reviewing Part 2.*

## Overview

**The question.** Part 2 classifies endpoints using embeddings from a modern, fine-tuned language model. How would a **classic text-classification pipeline**, the kind that was standard before embeddings became mainstream, do on the same task? This tells how much the modern embeddings actually add.

**How text became numbers before embeddings:**
- **Bag of words**: one number per word of the vocabulary, counting how often the word appears in the text (word order is lost, hence "bag").
- **TF-IDF**: the same counts, weighted by how rare each word is across all texts, so *"the"* counts little and *"seroconversion"* counts a lot (also used in Part 1A to find each category's distinctive words).
- **Character n-grams**: counting short chunks of characters (here 2 to 5 characters, e.g. *"ser"*, *"sero"*) instead of whole words, which copes better with abbreviations, variants and typos.

The resulting vectors are very long (one slot per word or chunk, tens of thousands) and mostly zeros, and they only match **identical words or chunks**: *"Cl/F"* and *"Total body clearance"* share none, so for TF-IDF they are unrelated, while the embedding model places them close together (Part 1A).

**Plan:**
1. The classic pipeline: TF-IDF on words and on character chunks, then logistic regression.
2. Its results on the same 5 grouped splits as Part 2, compared with Part 2's.
3. On one test set, the items where the two approaches disagree, to see *what* the embeddings add.
4. A combination of both.

### Setup

**What:** load the data and embeddings and rebuild **exactly the same data splits as Part 2** (same code, same seeds), so that the comparison is fair: every model is trained and tested on the same endpoints.

**Settings:** logistic regression uses the same settings as the Part 2 baseline (`C=10`, balanced class weights). They are not tuned here, so the test sets play no role in choosing them.

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_union

pd.set_option("display.max_colwidth", 70)
df = pd.read_csv("data/endpoints_data.csv")
X_emb = np.load("embeddings/anchor.npy").astype(np.float32)
CLASSES = df["case"].value_counts().index.tolist()
y_all = df["case"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
groups = df["anchor"].str.lower().str.strip().to_numpy()

def grouped_split(idx, n_splits, seed):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rest, held = next(sgkf.split(idx, y_all[idx], groups[idx]))
    return idx[rest], idx[held]

def make_splits(seed):                         # identical to Part 2, Section 1
    trval, test = grouped_split(np.arange(len(df)), n_splits=7, seed=seed)
    train, val = grouped_split(trval, n_splits=6, seed=seed)
    dedup = lambda ix: df.iloc[ix].drop_duplicates(["anchor", "case"]).index.to_numpy()
    return train, dedup(val), dedup(test)

train_idx, val_idx, test_idx = make_splits(42)
print(f"Seed 42: {len(train_idx)} training rows, {len(test_idx)} test endpoints (same as Part 2)")

Seed 42: 2438 training rows, 398 test endpoints (same as Part 2)


## 1. The classic pipeline

**What:** turn each anchor into two TF-IDF vectors, one over **words** and one over **character chunks of 2-5 characters** (within word boundaries), glue them together, and train a logistic regression on them.

**How:** `TfidfVectorizer` learns the vocabulary and the rarity weights from the **training texts only**, then converts any text into a vector; `make_union` concatenates the two vectors. The printout shows how long and how empty these vectors are.

In [2]:
def classic_features():
    return make_union(TfidfVectorizer(), TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)))

def classic_model():
    return LogisticRegression(C=10, max_iter=3000, class_weight="balanced")

def scores(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro")}

feats = classic_features().fit(df["anchor"].iloc[train_idx])
X_tr = feats.transform(df["anchor"].iloc[train_idx])
print(f"Each text becomes a vector of {X_tr.shape[1]:,} numbers; "
      f"on average only {X_tr.getnnz(axis=1).mean():.0f} of them are not zero")
print("Embeddings, for comparison: 768 numbers, all non-zero")

Each text becomes a vector of 29,354 numbers; on average only 93 of them are not zero
Embeddings, for comparison: 768 numbers, all non-zero


## 2. Results on the 5 grouped splits

**What:** train and test the classic pipeline on the same 5 splits as Part 2's robustness check (seeds 0-4), and on the seed-42 split used for Part 2's main evaluation.

**Reference results from Part 2** (Sections 9-10, same splits): the final neural network on embeddings reaches **73.9% ± 2.3 accuracy and 0.716 ± 0.030 macro-F1** over seeds 0-4, and 71.1% / 0.683 on seed 42; kNN on embeddings 68.5% / 0.672; logistic regression on embeddings 61.1% / 0.606 on seed 42.

In [3]:
def run_classic(seed):
    tr, _, te = make_splits(seed)
    f = classic_features().fit(df["anchor"].iloc[tr])
    m = classic_model().fit(f.transform(df["anchor"].iloc[tr]), y_all[tr])
    return m.predict(f.transform(df["anchor"].iloc[te])), te

rows = []
for seed in [0, 1, 2, 3, 4, 42]:
    pred, te = run_classic(seed)
    rows.append({"seed": seed, **scores(y_all[te], pred)})
res = pd.DataFrame(rows).set_index("seed")
summary = res.loc[[0, 1, 2, 3, 4]].agg(["mean", "std"])
print("Classic pipeline (TF-IDF words + characters -> logistic regression):")
print(res.round(3).to_string())
print("\nSeeds 0-4:")
summary.round(3)

Classic pipeline (TF-IDF words + characters -> logistic regression):
      accuracy  macro_f1
seed                    
0        0.689     0.650
1        0.724     0.723
2        0.662     0.619
3        0.672     0.625
4        0.694     0.633
42       0.688     0.661

Seeds 0-4:


,accuracy,macro_f1
mean,0.688,0.650
std,0.024,0.043


**Observations - classic pipeline**
- Each text becomes a vector of **29,354 numbers, of which on average only 93 are not zero** (first output): a huge, nearly empty space, one slot per word or character chunk seen in training.
- Over the 5 splits the classic pipeline reaches **68.8% ± 2.4 accuracy and 0.650 ± 0.043 macro-F1** (table above): about the level of kNN on embeddings (68.5% / 0.672, Part 2), and clearly below the Part 2 neural network (**73.9% / 0.716**). On the seed-42 split: 68.8% / 0.661, versus 71.1% / 0.683 for the network.
- It does much better than logistic regression on *embeddings* (61.1% on seed 42, Part 2): the same linear model works far better in the huge keyword space, where many categories can be told apart by a few keywords (Part 1A), than in the dense 768-number embedding space, where categories are not separable by straight boundaries (Part 1C).
- **The modern embeddings plus a neural network add about 5 points of accuracy and 7 points of macro-F1** over this classic approach.

## 3. What do the embeddings add?

**What:** on the seed-42 test set, compare item by item the classic pipeline with the Part 2 neural network (loaded from `models/endpoint_classifier.pt`, as in Bonus B), and look at the items where only one of the two is right.

**Hypothesis to test:** the embeddings should help most on texts whose words do not appear in training, such as paraphrases and abbreviations, where word matching has nothing to hold on to.

**How:** for each test text I measure its **word overlap with the training data**: the highest TF-IDF similarity (words only) between it and any training text, from 0 (no word in common with any training text) to 1 (an identical wording exists). Then I compare this overlap across the groups of items.

In [4]:
class EndpointClassifier(nn.Module):           # same definition as Part 2, to load its weights
    def __init__(self, in_dim=768, hidden=(512, 128), n_classes=9, dropout=0.3, input_dropout=0.1):
        super().__init__()
        layers, prev = [nn.Dropout(input_dropout)], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

saved = torch.load("models/endpoint_classifier.pt")
nn_model = EndpointClassifier(hidden=saved["config"]["hidden"])
nn_model.load_state_dict(saved["state_dict"]); nn_model.eval()
with torch.no_grad():
    nn_pred = nn_model(torch.from_numpy(X_emb[test_idx])).argmax(1).numpy()
classic_pred, _ = run_classic(42)
y_te = y_all[test_idx]

# Word overlap of each test text with the training texts
words = TfidfVectorizer().fit(df["anchor"].iloc[train_idx])
overlap = (words.transform(df["anchor"].iloc[test_idx]) @ words.transform(df["anchor"].iloc[train_idx]).T).max(axis=1)
overlap = np.asarray(overlap.todense()).ravel()

cmp = df.loc[test_idx, ["anchor", "case"]].assign(
    network=[CLASSES[k] for k in nn_pred], classic=[CLASSES[k] for k in classic_pred], word_overlap=overlap)
nn_ok, cl_ok = nn_pred == y_te, classic_pred == y_te
cmp["group"] = np.select([nn_ok & cl_ok, nn_ok & ~cl_ok, ~nn_ok & cl_ok], ["both right", "only network right",
                         "only classic right"], "both wrong")
print(f"Accuracy on this test set: network {nn_ok.mean():.3f} | classic {cl_ok.mean():.3f}")
cmp.groupby("group")["word_overlap"].agg(items="size", mean_word_overlap="mean",
                                          share_with_no_word_in_common=lambda s: (s == 0).mean()).round(3)

Accuracy on this test set: network 0.711 | classic 0.688


,items,mean_word_overlap,share_with_no_word_in_common
group,,,
both right,230,0.572,0.048
both wrong,71,0.413,0.239
only classic right,44,0.472,0.182
only network right,53,0.424,0.208


**What:** the items where only one approach is right, with the lowest word overlap first.

In [5]:
print("Only the NETWORK (embeddings) is right:")
print(cmp[cmp["group"] == "only network right"].sort_values("word_overlap")
      [["anchor", "case", "classic", "word_overlap"]].head(12).round(2).to_string(index=False))
print("\nOnly the CLASSIC pipeline is right:")
print(cmp[cmp["group"] == "only classic right"].sort_values("word_overlap")
      [["anchor", "case", "network", "word_overlap"]].head(12).round(2).to_string(index=False))

Only the NETWORK (embeddings) is right:
                                                anchor            case       classic  word_overlap
                                                   SBA Pharmacokinetic  General Case          0.00
                                            BAY73-4506 Pharmacokinetic  General Case          0.00
                                                CPAQ-R        COA Tool  General Case          0.00
                                                LYMQOL        COA Tool  General Case          0.00
                                      hydroxybupropion Pharmacokinetic  General Case          0.00
                                             MOS_4_new        COA Tool  General Case          0.00
                                                  MSPS        COA Tool  General Case          0.00
                      paracetamol absorption technique Pharmacokinetic  General Case          0.00
                                                TAPQOL        COA Too

**Observations - where the two approaches differ**
- On this test set the network is right on **53 items the classic pipeline gets wrong**, and the classic pipeline on **44 items the network gets wrong** (first table): the two make substantially different mistakes.
- **My hypothesis is not supported.** Items only the network gets right do not have notably lower word overlap than items only the classic pipeline gets right (mean 0.42 vs 0.47; share with no word in common with any training text: 21% vs 18%). The embeddings' advantage is not simply "handling texts with unseen words".
- The examples show a more specific pattern: on **short unknown acronyms and names** (word overlap 0), the two approaches fail in opposite directions:
  - the **classic pipeline defaults to General Case**, the most common category and the catch-all: it gets wrong unknown instruments (*"LYMQOL"*, *"TAPQOL"*, *"CPAQ-R"*) and drug names (*"BAY73-4506"*, *"hydroxybupropion"*), but right unknown lab measures that really are General Case (*"MDM2"*, *"MT-ATP6"*, *"Desmosterol"*);
  - the **network** recognises instrument acronyms and drug names, presumably because the embedding model learned them during fine-tuning (Bonus B found its largest gain on COA Tool), but **over-uses COA Tool for acronyms that are actually lab measures or genes** (*"MDM2"*, *"MT-ATP6"*, *"WSS"* predicted as COA Tool or Adverse Event).
- These complementary mistakes suggest that **combining the two** could help.

## 4. Combining both

**What:** give a single logistic regression **both** kinds of features: the TF-IDF vector and the 768 embedding numbers side by side.

**Why:** the two approaches make different mistakes (Section 3), so a model that sees both might get the best of each. It is also the cheapest combination to build: no new model type, just more input features.

**How:** the TF-IDF vector is stored in a memory-saving "sparse" format (only non-zero values); `scipy.sparse.hstack` glues it to the embeddings. Same 5 splits, same settings.

In [6]:
rows = []
for seed in [0, 1, 2, 3, 4, 42]:
    tr, _, te = make_splits(seed)
    f = classic_features().fit(df["anchor"].iloc[tr])
    both = lambda ix: sp.hstack([f.transform(df["anchor"].iloc[ix]), sp.csr_matrix(X_emb[ix])]).tocsr()
    m = classic_model().fit(both(tr), y_all[tr])
    rows.append({"seed": seed, **scores(y_all[te], m.predict(both(te)))})
combo = pd.DataFrame(rows).set_index("seed")
print("TF-IDF + embeddings -> logistic regression:")
print(combo.round(3).to_string())
print("\nSeeds 0-4, side by side:")
pd.DataFrame({"classic (TF-IDF)": res.loc[[0, 1, 2, 3, 4]].mean(),
              "TF-IDF + embeddings": combo.loc[[0, 1, 2, 3, 4]].mean()}).T.round(3)

TF-IDF + embeddings -> logistic regression:
      accuracy  macro_f1
seed                    
0        0.730     0.714
1        0.765     0.752
2        0.708     0.665
3        0.735     0.707
4        0.739     0.709
42       0.759     0.732

Seeds 0-4, side by side:


,accuracy,macro_f1
classic (TF-IDF),0.688,0.65
TF-IDF + embeddings,0.735,0.71


**Observations - combination**
- A single logistic regression fed **both** TF-IDF and embeddings reaches **73.5% accuracy and 0.710 macro-F1** over the 5 splits (table above), **the same level as the Part 2 neural network (73.9% / 0.716)**, and clearly above either kind of features alone with logistic regression (68.8% with TF-IDF; 61.1% with embeddings on seed 42). On the seed-42 split it reaches 75.9% / 0.732, above the network's 71.1% / 0.683.
- So the network's advantage over a linear model comes from being able to draw curved boundaries in the embedding space, and **adding the keyword features lets a plain linear model reach the same result**.

## Takeaway
- A classic, pre-embedding pipeline already gets **~69%**, because most categories are recognisable by their keywords (Part 1A). The fine-tuned embeddings with a small network add **~5 points**.
- **Keywords and embeddings make different mistakes**: keyword matching knows nothing about unfamiliar acronyms and falls back to General Case; the embeddings recognise many of them but over-use COA Tool for acronyms in general.
- **The simplest model that matches the neural network is a logistic regression on both kinds of features**: one standard, fast, easily explained model, with no network to train, tune or early-stop. In a real project I would test this on the validation sets and consider it as the production candidate, or feed the keyword features to the network as well, before adding any complexity.
- Caveat: as everywhere in this test, these are averages over 5 splits of ~400 endpoints; differences of 1 point are within the noise (the standard deviations above are 2-4 points).